## tl;dr

继续一个受控验证批次。6 个固定查询与 10 个未知值案例通过；26 条模拟历史变化全部取回。6 个指定详情响应比完整知识 JSON 的字节减少中位数为 91.5%。这不是 token、端到端耗时或市场需求验证。


## Context & Methods

这是报告的可重跑计算附件，粒度为项目 × 固定详情任务。原始测量代码：evaluations/knowledge-agent-tasks.ts；运行命令（仓库根目录）：npm run knowledge:evaluate -- after。HTTP handler 连接隔离内存 SQLite，未部署。固定声明评估时间为 2026-08-28T02:00:00Z。

### Key Assumptions

只比较同项目、同完整文档基线；短响应按任务选子集，不是无损压缩。预先知道 slug/id，未计发现与回退的请求。样板为 6 个记录／3 个项目组，外部检查定向选择 4 个记录。26 条变化、10 个未知值案例均为合成测试。不能估计全库覆盖率或真实用户成功率。

无需第三方 Python 包。随附 JSON/SQL 文件与本 notebook 放在同一目录。当前环境无 Jupyter 内核；本次用 Python 标准库按顺序执行所有 code cells 并保存输出，未验证 Jupyter UI。


## Data

### Load the preserved measurements


In [1]:
import json, sqlite3, statistics
from pathlib import Path
before_text = Path("before.json").read_text()
after_text = Path("after.json").read_text()
before, after = json.loads(before_text), json.loads(after_text)
assert before["evaluationTime"] == after["evaluationTime"]
assert before["population"] == after["population"]
assert len(after["payloads"]) == 6
print(json.dumps({"before": before["evaluatedAt"], "after": after["evaluatedAt"], "population": after["population"]}, ensure_ascii=False, indent=2))


{
  "before": "2026-08-28T05:03:01.675Z",
  "after": "2026-08-28T05:15:03.409Z",
  "population": {
    "projects": 162,
    "representative": [
      "vgpu",
      "opencode",
      "microduck",
      "microduck-runtime",
      "microduck-rl",
      "microduck-policies"
    ],
    "holdout": [
      "openhands",
      "langgraph",
      "lerobot",
      "playwright-mcp"
    ]
  }
}


### Reconcile paired measurements with the preserved SQL

payload-analysis.sql 的两个绑定参数分别为 before.json 与 after.json 原文。源字段是脚本实测 UTF-8 字节，SQL 计算成对减少比例并验证基线。


In [2]:
connection = sqlite3.connect(":memory:")
connection.row_factory = sqlite3.Row
query = Path("payload-analysis.sql").read_text()
chart_rows = [dict(row) for row in connection.execute(query, (before_text, after_text))]
connection.close()
assert len(chart_rows) == 18
assert all(row["beforeFullBytes"] == row["fullBytes"] for row in chart_rows)
detail_rows = [row for row in chart_rows if row["ordinal"] == 3]
assert len({row["project"] for row in detail_rows}) == 6
print(json.dumps(detail_rows, ensure_ascii=False, indent=2))


[
  {
    "project": "microduck",
    "question": "Locate the browser simulator and its stated resource terms",
    "legacyBytes": 13887,
    "fullBytes": 19837,
    "taskBytes": 1519,
    "beforeFullBytes": 19837,
    "reduction": 0.9234259212582547,
    "representation": "单任务 Knowledge JSON",
    "ordinal": 3,
    "bytes": 1519
  },
  {
    "project": "microduck-policies",
    "question": "Locate one policy artifact with scoped digest and resource terms",
    "legacyBytes": 16702,
    "fullBytes": 32822,
    "taskBytes": 2384,
    "beforeFullBytes": 32822,
    "reduction": 0.9273657912375846,
    "representation": "单任务 Knowledge JSON",
    "ordinal": 3,
    "bytes": 2384
  },
  {
    "project": "microduck-rl",
    "question": "Locate simulation assets with scoped version and evidence",
    "legacyBytes": 11375,
    "fullBytes": 15878,
    "taskBytes": 2328,
    "beforeFullBytes": 15878,
    "reduction": 0.8533820380400554,
    "representation": "单任务 Knowledge JSON",
    "ordinal": 3,

## Results

### Paired size reductions and completeness checks


In [3]:
metrics = {
 "minimum_task_bytes": min(row["taskBytes"] for row in detail_rows),
 "maximum_task_bytes": max(row["taskBytes"] for row in detail_rows),
 "median_reduction": statistics.median(row["reduction"] for row in detail_rows),
 "sum_full_bytes": sum(row["fullBytes"] for row in detail_rows),
 "sum_task_bytes": sum(row["taskBytes"] for row in detail_rows),
 "queries_correct": sum(row["results"] == row["expected"] for row in after["discovery"]),
 "queries_total": len(after["discovery"]),
 "unknown_cases_correct": sum(row["actual"] == row["expected"] for row in after["uncertainty"]),
 "unknown_cases_total": len(after["uncertainty"]),
 "history_before": before["history"]["exposedChanges"],
 "history_after": after["history"]["exposedChanges"],
 "history_unique": after["history"]["uniqueEvents"],
 "history_denominator": after["history"]["ledgerChanges"],
 "holdouts_without_interfaces": sum(row["interfaces"] == 0 for row in after["holdoutCoverage"]),
 "holdout_denominator": len(after["holdoutCoverage"])
}
assert metrics["queries_correct"] == metrics["queries_total"] == 6
assert metrics["unknown_cases_correct"] == metrics["unknown_cases_total"] == 10
assert metrics["history_after"] == metrics["history_unique"] == metrics["history_denominator"] == 26
assert metrics["history_before"] == 20
assert metrics["holdouts_without_interfaces"] == metrics["holdout_denominator"] == 4
assert all(row["taskBytes"] < row["legacyBytes"] < row["fullBytes"] for row in detail_rows)
print(json.dumps(metrics, ensure_ascii=False, indent=2))


{
  "minimum_task_bytes": 1519,
  "maximum_task_bytes": 2384,
  "median_reduction": 0.9152894367597806,
  "sum_full_bytes": 140476,
  "sum_task_bytes": 12938,
  "queries_correct": 6,
  "queries_total": 6,
  "unknown_cases_correct": 10,
  "unknown_cases_total": 10,
  "history_before": 20,
  "history_after": 26,
  "history_unique": 26,
  "history_denominator": 26,
  "holdouts_without_interfaces": 4,
  "holdout_denominator": 4
}


## Takeaways

本轮支持严格只读查询、按需证据与历史导航的可行性，不支持通用知识库或长期可靠性已经成立。下一轮应填补非样板记录，区分运行配置与能力，建立字段复核规则，再对比两个客户端的真实任务结果。报告中 25% 改善门槛与 30 任务／10 项目均为建议，不是本次实测结果。
